In [1]:
import pandas as pd
import os

# === CONFIGURATION ===
dir_path = 'TSP-BDa_Outer_300_1500_10-Toden-NCI'
files = [
    'CIBERSORTx_Results.txt',
    'NNLS_All-NCI_Counts_v46_Clean_UniquePatients_CPM.txt',
    'nuSVR_Counts_TSP-BDa_Outer_300_1500_10-Toden-NCI.txt',
    'QP_All-NCI_Counts_v46_Clean_UniquePatients_CPM_composition.txt',
    'TSP-BDa_Inner_100each_seed42_filtered_All-NCI_Counts_v46_Clean_UniquePatients_CPM_BayesPrism_renamed.txt',
    'TSP-HBA_Inner_100each_seed42-ReDeconv_Top1500_ReDeconv_results.tsv',
    'TSP-HBA_Inner_100each_seed42_Toden_NCI_MuSiC.txt'
]

method_map = ['CIBERSORTx', 'MuSiC', 'QP', 'NNLS', 'BayesPrism', 'nuSVR', 'ReDeconv']

def normalize_to_100(df):
    numeric_cols = df.select_dtypes(include='number').columns
    df[numeric_cols] = df[numeric_cols].div(df[numeric_cols].sum(axis=1), axis=0) * 100
    return df

def clean_dataframe(df, file):
    if (
        file.endswith('BayesPrism_renamed.txt') or
        file.endswith('MuSiC.txt') or
        file.startswith('CIBERSORTx_Results') or
        file.endswith('ReDeconv_results.tsv')
    ):
        if file.startswith('CIBERSORTx_Results'):
            df = df.drop(columns=['P-value', 'Correlation', 'RMSE'], errors='ignore')
        df = normalize_to_100(df)

    elif file.startswith('QP') or file.startswith('NNLS'):
        df = df.drop(columns=[
            'RMSE-Composition', 'r-Composition',
            'RMSE-PredictedCounts', 'r-PredictedCounts'
        ], errors='ignore')

    return df.round(5)

def detect_method(filename):
    return next((m for m in method_map if m in filename), None)

# === FIRST PASS: COLLECT ALL CELL TYPES ===
all_celltypes = set()
temp_dfs = []
for file in files:
    file_path = os.path.join(dir_path, file)
    df = pd.read_csv(file_path, sep='\t', index_col=0)
    df = clean_dataframe(df, file)
    all_celltypes.update(df.columns)  # keep full set
    temp_dfs.append((file, df))

# Remove metadata columns from all_celltypes
all_celltypes = {c for c in all_celltypes if c not in ['Method', 'Sample']}

# === SECOND PASS: ENSURE CONSISTENT COLUMNS ===
all_results = []
for file, df in temp_dfs:
    # reindex to ensure all cell types present
    df = df.reindex(columns=list(all_celltypes))  # fill_value=None (default)

    method = detect_method(file)
    df['Method'] = method
    df['Sample'] = df.index
    all_results.append(df.reset_index(drop=True))

# === MERGE AND SAVE ===
merged_df = pd.concat(all_results, ignore_index=True)
output_file = os.path.join(dir_path, 'merged_normalised_results_Toden_NCI.txt')
merged_df.to_csv(output_file, sep='\t', index=False)
print(f"Merged results saved to: {output_file}")


Merged results saved to: TSP-BDa_Outer_300_1500_10-Toden-NCI/merged_normalised_results_Toden_NCI.txt
